https://colab.research.google.com/drive/1EM43Kwu80skT3pPOxhyB7NhMyfdXxvgo?usp=sharing

#Fine Tuning
# Scenario: AI System for Detecting Lung Diseases from Chest X-rays
 🚨 The Problem

 A large hospital receives thousands of chest X-rays daily.

 Radiologists are:

 Overworked

 Limited in number

 Required to make fast decisions

 Sometimes critical conditions like:

 👉 Pneumonia
 👉 COVID-19
 👉 Normal lungs

 must be identified within minutes.

 Delays can cost lives.

 💡 The Solution: Build an AI Assistant

 The hospital decides to deploy an AI model that can pre-screen X-rays and
 alert doctors.

 But there is a challenge…

 ❗ Medical datasets are usually SMALL.

 Training a deep neural network from scratch would require:

 Millions of labeled X-rays

 Massive GPU clusters

 Months of training

 Not practical.

 ⭐ Enter Transfer Learning (Your Code)

 Instead of starting from zero, engineers use:

 👉 ResNet50 trained on ImageNet

 Although ImageNet contains everyday objects (dogs, cars, etc.),
 the early CNN layers learn universal visual patterns, like:

 ✅ Edges
 ✅ Gradients
 ✅ Textures
 ✅ Shapes

 These features are also present in medical scans.

In [1]:
import torch

In [2]:
import torchvision.models as models



In [3]:
# Import neural network module from PyTorch
from torch import nn

In [4]:
model = models.resnet50(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 166MB/s]


In [5]:
for param in model.parameters():
    param.requires_grad = False

In [6]:
num_classes = 3

In [7]:
model.fc = nn.Linear(model.fc.in_features, num_classes)

In [8]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Trainable params: {trainable_params}")

Trainable params: 6147


# Scenario: AI System for Detecting Fraud in Financial Transactions
🚨 The Problem
 A major bank processes millions of transactions daily.
 Fraud analysts are:
 - Overwhelmed by the sheer volume
 - Limited in number
 - Required to make instant decisions
 - Missing subtle fraud patterns hidden in massive datasets
 Critical cases like:
 👉 Credit card fraud
 👉 Money laundering
 👉 Normal transactions
 must be flagged in real-time.
 Delays can cost millions and damage customer trust.

In [13]:
import torch
from torch import nn
from transformers import BertModel, BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased")

for param in bert.parameters():
    param.requires_grad = False

class FraudClassifier(nn.Module):
    def __init__(self, bert_model, num_classes):
        super(FraudClassifier, self).__init__()
        self.bert = bert_model
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.pooler_output
        return self.fc(cls_output)

num_classes = 3
model = FraudClassifier(bert, num_classes)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable_params}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable params: 2307


In [14]:
# Install required libraries if not already installed:
# pip install transformers datasets peft accelerate

# Install required libraries if not already installed:
# pip install transformers datasets peft accelerate

from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset
from peft import get_peft_model, LoraConfig

# 1. Load base model and tokenizer
model_name = "gpt2"   # you can replace with another causal LM
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Ensure pad token exists (GPT-2 doesn't have one by default)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Define LoRA configuration
lora_config = LoraConfig(
    r=8,                  # rank of low-rank matrices
    lora_alpha=32,        # scaling factor
    lora_dropout=0.1,     # dropout for regularization
    bias="none",          # no bias terms updated
    task_type="CAUSAL_LM",# language modeling task
    target_modules=["c_attn"]  # GPT-2 attention projection layers
)

# Wrap model with LoRA adapters
model = get_peft_model(model, lora_config)

# 3. Create a tiny synthetic dataset
train_texts = ["Hello world", "LoRA fine-tuning is fun", "Transformers are powerful"]
eval_texts = ["Testing evaluation", "Another sample"]

train_dataset = Dataset.from_dict({"text": train_texts})
eval_dataset = Dataset.from_dict({"text": eval_texts})

# Tokenization function with labels
def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )
    tokens["labels"] = tokens["input_ids"].copy()  # labels required for loss
    return tokens

# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# 4. Training configuration
training_args = TrainingArguments(
    output_dir="./lora-finetuned-model",   # save directory
    per_device_train_batch_size=2,         # small batch size
    gradient_accumulation_steps=2,         # accumulate gradients
    learning_rate=2e-4,                    # tuned for LoRA
    num_train_epochs=1,                    # demo run
    logging_steps=1,                       # log every step
    save_strategy="epoch",                 # save at end of epoch
    fp16=True                              # mixed precision training
)

# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

# 6. Train
trainer.train()

# 7. Save only LoRA adapter weights (few MB instead of GBs!)
model.save_pretrained("./lora-weights")

print("Training complete. LoRA weights saved in ./lora-weights")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,8.129440


Training complete. LoRA weights saved in ./lora-weights


#Scenario: Fraud Detection in Customer Support Chats
- Problem: Fraudsters often contact bank support pretending to be customers, trying to reset passwords or gain access.
- Challenge: Analysts can’t manually read millions of chat transcripts.
- Solution: Use LoRA fine‑tuning on a pretrained language model (like distilbert-base-uncased) to classify chats as:
- 0 → Normal inquiry
- 1 → Fraud attempt
- 2 → Suspicious but unclear

In [15]:
import torch
from torch import nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin","v_lin"]
)

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable_params}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable params: 740355
